# v4: Probability calibration

**Version:** v4  
**Kaggle submission:** **#4** `sub04-2026-11-20-v4_platt` → `outputs/submission_v4_platt.csv` (public score 23,875.67). Isotonic variant built but not uploaded.

**Hypothesis:** the flag rule depends on $p$ itself (flag when $p\mu>12.5$), so miscalibrated probabilities cost money. Calibrate the raw probabilities (Platt or isotonic) before applying the v3 decision rule.

*Self-contained: needs only `../data/train.csv` and `../data/test.csv`. Part of the Fleet Maintenance project (Kaggle `mlprojectfleet2`); the cost, decision rule and validation protocol are derived in `fleet_maintenance.ipynb`.*

In [1]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, xgboost as xgb
from pathlib import Path
from scipy.stats import ks_2samp
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score, r2_score, brier_score_loss, log_loss
from sklearn.model_selection import StratifiedKFold, KFold, train_test_split, cross_val_predict

DATA, OUT = Path("../data"), Path("outputs"); OUT.mkdir(exist_ok=True)
SEED = 42
INSPECT, UNDER, OVER = 10.0, 0.20, 0.02          # inspection cost; shortage cost / unit; unused-budget cost / unit
Q_STAR = UNDER / (UNDER + OVER)                  # newsvendor critical ratio 0.909
HISTB = [f"s{i:03d}" for i in range(54, 64)]     # 10-bin histogram that is shifted by one bin in test

def load(raw=False):
    """train/test with the v1 fixes: realign s054-s063 (test bin k+1 == train bin k), drop s047 (all NaN in
    test) and s146 (constant). raw=True returns the files untouched."""
    tr, te = pd.read_csv(DATA / "train.csv"), pd.read_csv(DATA / "test.csv")
    if not raw:
        te[HISTB[:-1]] = te[HISTB[1:]].to_numpy(); te[HISTB[-1]] = np.nan
        tr, te = tr.drop(columns=["s047", "s146"]), te.drop(columns=["s047", "s146"])
    return tr, te, [c for c in tr.columns if c.startswith("s")]

tr, te, S = load()
dev_idx, ho_idx = train_test_split(np.arange(len(tr)), test_size=0.2, stratify=tr.failure, random_state=SEED)
dev, ho = tr.iloc[dev_idx].reset_index(drop=True), tr.iloc[ho_idx].reset_index(drop=True)   # ho: locked 20% hold-out
y, c = dev.failure.to_numpy(), dev.repair_cost.fillna(0).to_numpy() * dev.failure.to_numpy()
y_ho, c_ho = ho.failure.to_numpy(), ho.repair_cost.fillna(0).to_numpy() * ho.failure.to_numpy()
c_all = tr.repair_cost.fillna(0).to_numpy() * tr.failure.to_numpy()
F = list(StratifiedKFold(5, shuffle=True, random_state=SEED).split(np.zeros(len(y)), y))   # the 5 fixed folds
print(f"train {tr.shape}  test {te.shape}  sensors {len(S)}  dev {len(dev)} / hold-out {len(ho)}  failures {int(y.sum())}")

train (56000, 172)  test (20000, 169)  sensors 168  dev 44800 / hold-out 11200  failures 318


In [2]:
def vehicle_cost(c, flag, b):
    return np.where(flag == 1, INSPECT + UNDER * np.maximum(c - b, 0) + OVER * np.maximum(b - c, 0), c)

def per_vehicle(c, flag, b):
    return float(vehicle_cost(np.asarray(c, float), np.asarray(flag), np.asarray(b, float)).mean())

fold_cost = lambda flag, b: np.array([per_vehicle(c[v], flag[v], b[v]) for _, v in F])

def write_submission(ids, flag, budget, name):
    flag, budget = np.asarray(flag).astype(int), np.maximum(np.asarray(budget, float), 0)
    assert len(ids) == 20000 and ids.is_unique and set(flag) <= {0, 1} and np.isfinite(budget).all()
    pd.DataFrame({"id": ids, "flag": flag, "budget": budget.round(2)}).to_csv(OUT / f"submission_{name}.csv", index=False)
    print(f"wrote outputs/submission_{name}.csv: flagged {flag.mean():.2%} of test vehicles")

def decide_global(p, fail_costs):
    """Same repair-cost distribution F (empirical failure costs) for every vehicle."""
    p = np.clip(np.asarray(p, float), 0, 1); fc = np.sort(np.asarray(fail_costs, float)); mu = fc.mean()
    b = np.zeros_like(p); need = p > 1 - Q_STAR
    b[need] = np.quantile(fc, 1 - (1 - Q_STAR) / p[need])                       # newsvendor budget
    fail = (UNDER * np.maximum(fc[None, :] - b[:, None], 0) + OVER * np.maximum(b[:, None] - fc[None, :], 0)).mean(1)
    return ((INSPECT + p * fail + (1 - p) * OVER * b) < p * mu).astype(int), b   # flag iff cheaper than not flagging

def decide_pv(p, m, resid, fail_costs=None, K=99, scale=None):
    """Per-vehicle cost distribution: log c | fail ~ m_i + scale_i * residuals (out-of-fold residuals).
    fail_costs given -> budget from that GLOBAL distribution (ablation); scale -> per-vehicle spread (v9)."""
    p, m = np.clip(np.asarray(p, float), 0, 1), np.asarray(m, float)
    sc = np.ones_like(m) if scale is None else np.asarray(scale, float)
    M = np.exp(m[:, None] + sc[:, None] * np.quantile(resid, (np.arange(K) + 0.5) / K)[None, :]); mu = M.mean(1)
    b = np.zeros_like(p); need = p > 1 - Q_STAR; level = 1 - (1 - Q_STAR) / p[need]
    b[need] = (np.exp(m[need] + sc[need] * np.quantile(resid, level)) if fail_costs is None
               else np.quantile(np.asarray(fail_costs, float), level))
    fail = (UNDER * np.maximum(M - b[:, None], 0) + OVER * np.maximum(b[:, None] - M, 0)).mean(1)
    return ((INSPECT + p * fail + (1 - p) * OVER * b) < p * mu).astype(int), b

class Platt:
    def fit(self, p, y):
        self.m = LogisticRegression(C=1e6, max_iter=1000).fit(self._x(p), y); return self
    def predict(self, p): return self.m.predict_proba(self._x(p))[:, 1]
    @staticmethod
    def _x(p): p = np.clip(p, 1e-6, 1 - 1e-6); return np.log(p / (1 - p))[:, None]

class Iso:
    def fit(self, p, y):
        self.m = IsotonicRegression(out_of_bounds="clip", y_min=0, y_max=1).fit(p, y); return self
    def predict(self, p): return self.m.predict(p)

make_cal = lambda kind: {"platt": Platt, "iso": Iso}[kind]()

PARAMS = dict(n_estimators=300, max_depth=4, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8,
              min_child_weight=1, n_jobs=-1, random_state=SEED)                         # failure classifier (v2..)
CM = dict(n_estimators=200, max_depth=3, learning_rate=0.03, subsample=0.8, colsample_bytree=0.5,
          min_child_weight=5, n_jobs=-1, random_state=SEED)                             # repair-cost model (v5)
inner = KFold(5, shuffle=True, random_state=SEED)

def cost_model(params, Xa, lc, Xb):
    """XGBoost on log repair cost (failures only). Returns predictions for Xb and OUT-OF-FOLD residuals of Xa."""
    resid = lc - cross_val_predict(xgb.XGBRegressor(**params), Xa, lc, cv=inner)
    return xgb.XGBRegressor(**params).fit(Xa, lc).predict(Xb), resid

def fit_oof(X, params=PARAMS):
    """5-fold out-of-fold failure probabilities on dev, and probabilities for test from a model fit on all train."""
    o = np.zeros(len(dev))
    for a, b in F:
        o[b] = xgb.XGBClassifier(**params).fit(X.iloc[a], y[a]).predict_proba(X.iloc[b])[:, 1]
    return o

shift = lambda p, k: k * p / (1 - p) / (1 + k * p / (1 - p))       # prior shift by odds factor k

In [3]:
oof = fit_oof(dev[S])
p_te = xgb.XGBClassifier(**PARAMS).fit(tr[S], tr.failure).predict_proba(te[S])[:, 1]
print("reliability on OOF predictions (quantile bins of the raw p): predicted vs observed failure rate")
edges = np.quantile(oof, [0, .5, .8, .9, .95, .98, .99, 1.0])
for lo, hi in zip(edges[:-1], edges[1:]):
    m = (oof >= lo) & (oof <= hi); print(f"  p in [{lo:.4f}, {hi:.4f}]  n={m.sum():5d}  predicted {oof[m].mean():.4f}  observed {y[m].mean():.4f}")

reliability on OOF predictions (quantile bins of the raw p): predicted vs observed failure rate
  p in [0.0000, 0.0000]  n=22400  predicted 0.0000  observed 0.0002
  p in [0.0000, 0.0001]  n=13440  predicted 0.0001  observed 0.0000
  p in [0.0001, 0.0004]  n= 4480  predicted 0.0002  observed 0.0000
  p in [0.0004, 0.0027]  n= 2240  predicted 0.0010  observed 0.0027
  p in [0.0027, 0.0284]  n= 1344  predicted 0.0097  observed 0.0253
  p in [0.0284, 0.1484]  n=  448  predicted 0.0669  observed 0.1384
  p in [0.1484, 0.9911]  n=  448  predicted 0.4762  observed 0.4732


### Cross-fitted comparison (paired per fold)
Each fold's calibrator is fitted on the **other** folds' out-of-fold predictions, so no fold is calibrated on itself.

In [4]:
def cv_costs(kind):
    pc, costs = oof.copy(), []
    for a, b in F:
        if kind != "raw": pc[b] = make_cal(kind).fit(oof[a], y[a]).predict(oof[b])
        fl, bu = decide_global(pc[b], c[a][c[a] > 0]); costs.append(per_vehicle(c[b], fl, bu))
    return np.array(costs), pc
res = {k: cv_costs(k) for k in ("raw", "platt", "iso")}
print("method    CV cost/veh      paired delta vs raw (per fold)          folds better   Brier     log-loss")
for k, (costs, pc) in res.items():
    d = costs - res["raw"][0]
    print(f"{k:6s}  {costs.mean():.4f}±{costs.std():.3f}   {str(np.round(d, 3)):38s}  {int((d < 0).sum())}/5      {brier_score_loss(y, pc):.5f}   {log_loss(y, np.clip(pc, 1e-6, 1 - 1e-6)):.5f}")

method    CV cost/veh      paired delta vs raw (per fold)          folds better   Brier     log-loss
raw     1.2419±0.387   [0. 0. 0. 0. 0.]                        0/5      0.00464   0.01676
platt   1.1073±0.220   [-0.048 -0.17   0.005 -0.003 -0.455]    4/5      0.00452   0.01609
iso     1.0523±0.258   [-0.199 -0.064 -0.17  -0.031 -0.483]    5/5      0.00447   0.01607


### Hold-out check (11,200 vehicles, 79 failures): the first of the three planned uses of the locked hold-out

In [5]:
p_ho = xgb.XGBClassifier(**PARAMS).fit(dev[S], y).predict_proba(ho[S])[:, 1]
print(f"hold-out: n={len(ho)}, failures={y_ho.sum()}")
for k in ("raw", "platt", "iso"):
    pc = p_ho if k == "raw" else make_cal(k).fit(oof, y).predict(p_ho)
    fl, bu = decide_global(pc, c[c > 0]); print(f"  {k:6s} {per_vehicle(c_ho, fl, bu):.4f} per vehicle   flagged {fl.mean():.2%}")

hold-out: n=11200, failures=79
  raw    0.9060 per vehicle   flagged 2.28%
  platt  0.9547 per vehicle   flagged 3.29%
  iso    1.0079 per vehicle   flagged 3.87%


### Submissions: calibrator fitted on all dev OOF predictions, applied to the test probabilities

In [6]:
for k in ("platt", "iso"):
    pt = make_cal(k).fit(oof, y).predict(p_te)
    fl, bu = decide_global(pt, c_all[c_all > 0]); print(f"{k}: mean test p {pt.mean():.4f} (raw {p_te.mean():.4f}); ", end="")
    write_submission(te.id, fl, bu, f"v4_{k}")

platt: mean test p 0.0203 (raw 0.0183); wrote outputs/submission_v4_platt.csv: flagged 6.69% of test vehicles


iso: mean test p 0.0198 (raw 0.0183); wrote outputs/submission_v4_iso.csv: flagged 7.69% of test vehicles


## Results and caveats

The raw model is **under-confident exactly where decisions are made**: for raw p between 0.003 and 0.028 the observed failure rate is 2.5% against 1.0% predicted, and for 0.028-0.15 it is 13.8% against 6.7%. Above 0.15 it is well calibrated. So raw p under-flags vehicles near the 2.5% threshold.

| Method | CV cost / vehicle | Paired Δ vs raw | Folds better | Hold-out | Public |
|---|---|---|---|---|---|
| raw (v3) | 1.242 ± 0.387 | n/a | n/a | 0.906 | 27,760 |
| **Platt** | **1.107 ± 0.220** | −0.135 | 4/5 | 0.955 | **23,876** |
| isotonic | 1.052 ± 0.258 | −0.190 | 5/5 | 1.008 | not uploaded |

- **CV and hold-out disagreed** (the hold-out has only 79 failures: 0.05-0.10 is one or two large failures, so it is noise, but it is not support either). The **public score then favoured calibration** (27,760 → 23,876, −14%), and the same pattern repeated in v5 (raw 26,061 vs Platt 22,969). Isotonic is riskier with few positives; Platt is smoother and was kept.

**Caveats:** calibration is fitted to the *training* depot's failure rate. On test the model already predicts ~3× the failures; calibration raises that further (mean p 0.018 → 0.020) and test flag rate from 5.4% to 6.7% (Platt). That the public score improved suggests the test depot really does fail more often (the subject of v7). Test probabilities come from a model fit on all data while calibrators were fit on OOF predictions of 80%-data models, so test p is slightly more confident than what the calibrator saw. The hold-out must not drive tuning any more.